# Exploration vs. exploitation in written mathematical reasoning
### Humans vs. a reasoning model on the Nemotron-Math-HumanReasoning dataset

*Draft: text cells are written for you to adapt in your own words.*

## 1–2. The dataset

**What is it?** [Nemotron-Math-HumanReasoning](https://huggingface.co/datasets/nvidia/Nemotron-Math-HumanReasoning) is a small English text dataset released by NVIDIA on Hugging Face (created June 2025, licence CC BY-NC 4.0). It contains **50 competition-style math problems** (functional equations, number theory, inequalities, combinatorics, …), taken from NVIDIA's larger *OpenMathReasoning* dataset, which in turn was collected from the Art of Problem Solving (AoPS) forums.

Each problem has **five long-form solutions (250 rows in total)**:
- four written by **students with Olympiad experience** (`human_stage1` – `human_stage4`), and
- one generated by the reasoning model **QwQ-32B-Preview** (`qwq_32b_preview`).

The human authors were asked to write in a "thinking out loud" style that imitates reasoning models such as DeepSeek-R1, including false starts, guesses and checks. Fields: `problem`, `solution`, `expected_answer`, `solution_type`. The dataset accompanies the paper *"Is Human-Written Data Enough? The Challenge of Teaching Reasoning to LLMs Without RL or Distillation"* (arXiv 2507.09850). The dataset card does not explain what distinguishes the four human stages; the paper does.

**Possible ethical concerns**
- **Authorship and consent.** The solutions were written by students; the card does not say how they were recruited, paid or credited. The problems themselves come from AoPS forum posts written by users who did not explicitly agree to their reuse in AI training data.
- **"Human reasoning" is staged.** The students imitated a model's style, so the texts are not spontaneous thinking. Treating them as evidence of how humans *naturally* reason would be an overclaim; this matters for how I interpret any human–model difference.
- **Purpose of the data.** The dataset exists to train language models to imitate human-like reasoning. Analysing it is low risk, but results could be used to make model output harder to distinguish from human work (e.g. in education settings).
- **Licence.** Non-commercial use only, which is compatible with coursework.
- **Personal data.** The texts contain mathematics, not personal information, so privacy risk is low.

## 3. Research question

> **RQ: Do human-written solutions show more *exploration* (trying and switching between ideas) than model-generated solutions, or more *exploitation* (refining one line of attack)?**

Sub-questions:
1. **Lexical:** Do humans use more *exploration language* ("maybe", "let's try", "alternatively", "wait") relative to *exploitation language* ("therefore", "thus", "substituting", "we get")?
2. **Semantic:** When a solution is viewed as a sequence of steps, do human solutions make bigger *jumps* between consecutive steps and visit more *new* content?

**Plan**
1. Split every solution into **steps** (chunks of lines of at least 15 words).
2. **Lexical measure:** count exploration and exploitation marker words per solution and compute the *exploration share* = explore / (explore + exploit).
3. **Semantic measure:** represent each step as a **TF-IDF vector** and compute, per solution,
   - *step distance*: mean cosine distance between consecutive steps (large = jumping between ideas),
   - *novelty*: for each step, 1 − its highest similarity to any earlier step (high = new territory).
4. **Baseline:** shuffle the steps of each solution and recompute step distance. If the real order is no different from a random order, the metric is not capturing the *flow* of reasoning.
5. **Compare** human vs. model with a **paired test per problem** (Wilcoxon signed-rank), since every problem has both kinds of solution.
6. **Later (not done here):** replace TF-IDF with sentence embeddings, which capture meaning beyond shared words, and validate the metrics against a hand-labelled sample of steps.

### Pipeline
```
 solution text ──► split into steps ──┬──► marker-word counts ─────► exploration share
                                      │
                                      └──► TF-IDF vectors ──► cosine similarities ──► step distance, novelty
                                                                                        (vs. shuffled baseline)
                                                     ▼
                                   compare human vs. model per problem (Wilcoxon)
```

## 4. Analysis
### Setup and loading

In [ ]:
# !pip install datasets pandas scikit-learn scipy matplotlib
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import wilcoxon

RNG = np.random.default_rng(42)
COLORS = {"human": "#2a78d6", "model": "#eb6834"}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "figure.dpi": 110})

In [ ]:
df = load_dataset("nvidia/Nemotron-Math-HumanReasoning", split="train").to_pandas()

df["source"] = np.where(df["solution_type"].str.startswith("human"), "human", "model")
df["problem_id"] = df.groupby("problem").ngroup()
df["n_words"] = df["solution"].str.split().str.len()
print(df.shape)
df[["problem_id", "solution_type", "n_words", "solution"]].head()

### Overview: how long are the solutions?

In [ ]:
overview = (df.groupby("solution_type")
              .agg(solutions=("solution", "size"),
                   problems=("problem_id", "nunique"),
                   median_words=("n_words", "median"),
                   mean_words=("n_words", "mean"))
              .round(0).astype(int))
overview

### Step 1: split each solution into steps
Lines are joined until a chunk has at least `MIN_WORDS` words, so a one-line equation is kept together with the sentence that explains it.

In [ ]:
MIN_WORDS = 15

def split_steps(text, min_words=MIN_WORDS):
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    steps, buffer = [], ""
    for line in lines:
        buffer = f"{buffer} {line}".strip()
        if len(buffer.split()) >= min_words:
            steps.append(buffer)
            buffer = ""
    if buffer:                       # leftover short text joins the last step
        if steps:
            steps[-1] += " " + buffer
        else:
            steps.append(buffer)
    return steps

df["steps"] = df["solution"].apply(split_steps)
df["n_steps"] = df["steps"].str.len()
df.groupby("solution_type")["n_steps"].describe().round(1)

### Step 2: lexical markers of exploration vs. exploitation
Small hand-made word lists. They are a crude, transparent first measure, and their weaknesses are discussed in section 5.

In [ ]:
EXPLORE = [r"alternatively", r"another (?:approach|way|idea|method)", r"tr(?:y|ying|ied)", r"maybe",
           r"perhaps", r"what if", r"instead", r"wait", r"hmm+", r"guess", r"wonder", r"not sure",
           r"doesn'?t work", r"suppose"]
EXPLOIT = [r"therefore", r"thus", r"hence", r"it follows", r"substitut\w*", r"simplif\w*",
           r"which gives", r"we get", r"we obtain", r"plug\w* in", r"comput\w*", r"calculat\w*",
           r"conclude\w*"]

explore_re = re.compile(r"\b(?:" + "|".join(EXPLORE) + r")\b", re.IGNORECASE)
exploit_re = re.compile(r"\b(?:" + "|".join(EXPLOIT) + r")\b", re.IGNORECASE)

df["n_explore"] = df["solution"].str.count(explore_re)
df["n_exploit"] = df["solution"].str.count(exploit_re)
df["explore_per_1k"] = 1000 * df["n_explore"] / df["n_words"]
df["exploit_per_1k"] = 1000 * df["n_exploit"] / df["n_words"]
df["explore_share"] = df["n_explore"] / (df["n_explore"] + df["n_exploit"])

df.groupby("solution_type")[["explore_per_1k", "exploit_per_1k", "explore_share"]].mean().round(2)

### Step 3: TF-IDF trajectories
Each step becomes a TF-IDF vector. LaTeX commands (`\frac`, `\sum`, …) are removed and only alphabetic tokens are kept, so the vectors describe the *words* of a step rather than its formatting.

In [ ]:
steps = (df[["problem_id", "solution_type", "source", "steps"]]
         .explode("steps")
         .rename(columns={"steps": "text"}))
steps["sol_id"] = steps.index                      # index of the solution in df
steps["pos"] = steps.groupby(level=0).cumcount()   # position of the step in its solution
steps = steps.reset_index(drop=True)

def clean(text):
    text = re.sub(r"\\[a-zA-Z]+", " ", text)       # drop LaTeX commands
    return text.lower()

vectorizer = TfidfVectorizer(preprocessor=clean, stop_words="english", min_df=2,
                             token_pattern=r"(?u)\b[a-z]{2,}\b")
X = vectorizer.fit_transform(steps["text"])
print(f"{X.shape[0]} steps x {X.shape[1]} vocabulary terms")

In [ ]:
def trajectory_metrics(Xs):
    # Mean consecutive-step distance and mean novelty for one solution.
    n = Xs.shape[0]
    if n < 3:
        return np.nan, np.nan
    S = cosine_similarity(Xs)
    step_distance = 1 - np.diag(S, k=1)                       # step i vs. step i+1
    novelty = [1 - S[i, :i].max() for i in range(1, n)]       # step i vs. all earlier steps
    return step_distance.mean(), np.mean(novelty)

N_SHUFFLES = 20
rows = []
for sol_id, g in steps.groupby("sol_id"):
    Xs = X[g.index.to_numpy()]
    Xs = Xs[Xs.getnnz(axis=1) > 0]       # drop steps with no vocabulary words (pure formulas)
    dist, nov = trajectory_metrics(Xs)
    shuffled = [trajectory_metrics(Xs[RNG.permutation(Xs.shape[0])])[0] for _ in range(N_SHUFFLES)]
    rows.append({"sol_id": sol_id, "step_distance": dist, "novelty": nov,
                 "step_distance_shuffled": np.nanmean(shuffled) if Xs.shape[0] >= 3 else np.nan})

df = df.join(pd.DataFrame(rows).set_index("sol_id"))
METRICS = ["explore_share", "step_distance", "novelty"]
df.groupby("solution_type")[METRICS + ["step_distance_shuffled"]].mean().round(3)

### Step 4: paired comparison, human vs. model
For each problem, the four human solutions are averaged and compared to the one model solution. The Wilcoxon signed-rank test asks whether the per-problem differences are systematically above or below zero.

In [ ]:
paired = df.groupby(["problem_id", "source"])[METRICS + ["n_words"]].mean().unstack("source")

results = []
for m in METRICS + ["n_words"]:
    sub = paired[m].dropna()
    stat, p = wilcoxon(sub["human"], sub["model"])
    results.append({"metric": m, "problems": len(sub),
                    "human (median)": sub["human"].median(), "model (median)": sub["model"].median(),
                    "human > model (share of problems)": (sub["human"] > sub["model"]).mean(),
                    "Wilcoxon p": p})
results = pd.DataFrame(results).set_index("metric").round(3)
results

In [ ]:
# Does the order of steps matter? Real vs. shuffled step distance
order = df.groupby("source")[["step_distance", "step_distance_shuffled"]].mean().round(3)
order["real - shuffled"] = (order["step_distance"] - order["step_distance_shuffled"]).round(3)
order

## 6. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
titles = {"explore_share": "Exploration share of marker words",
          "step_distance": "Mean distance between consecutive steps",
          "novelty": "Mean novelty of steps"}

for ax, m in zip(axes, METRICS):
    data = [df.loc[df["source"] == s, m].dropna() for s in ["human", "model"]]
    bp = ax.boxplot(data, widths=0.5, patch_artist=True, showfliers=False,
                    medianprops={"color": "black", "linewidth": 1.5})
    for patch, s in zip(bp["boxes"], ["human", "model"]):
        patch.set_facecolor(COLORS[s]); patch.set_alpha(0.35); patch.set_edgecolor(COLORS[s])
    for i, (d, s) in enumerate(zip(data, ["human", "model"]), start=1):   # individual solutions
        ax.scatter(i + RNG.uniform(-0.12, 0.12, len(d)), d, s=10, color=COLORS[s], alpha=0.6, zorder=3)
    ax.set_xticks([1, 2], [f"Human (n={len(data[0])})", f"Model (n={len(data[1])})"])
    ax.set_title(titles[m], fontsize=10)

fig.suptitle("Exploration measures: human-written vs. QwQ-32B-Preview solutions", fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
# Trajectory of one problem: how far does each step move from the previous one?
example = df.loc[df["n_steps"] >= 8, "problem_id"].value_counts().index[0]
fig, ax = plt.subplots(figsize=(9, 4))
labelled = set()

for _, row in df[df["problem_id"] == example].iterrows():
    Xs = X[steps.index[steps["sol_id"] == row.name].to_numpy()]
    Xs = Xs[Xs.getnnz(axis=1) > 0]
    if Xs.shape[0] < 3:
        continue
    d = 1 - np.diag(cosine_similarity(Xs), k=1)
    x = np.linspace(0, 1, len(d))
    is_model = row["source"] == "model"
    label = {"human": "Human (4 solutions)", "model": "Model (QwQ-32B-Preview)"}[row["source"]]
    ax.plot(x, d, color=COLORS[row["source"]], linewidth=2.5 if is_model else 1.5,
            alpha=1 if is_model else 0.6, label=None if label in labelled else label)
    labelled.add(label)

ax.set_xlabel("Position in solution (0 = start, 1 = end)")
ax.set_ylabel("Cosine distance to previous step")
ax.set_ylim(0, 1.05)
ax.set_title(f"Problem {example}: " + df.loc[df['problem_id'] == example, 'problem'].iloc[0][:80] + "…", fontsize=10)
ax.legend(fontsize=9, ncol=2, loc="lower center", bbox_to_anchor=(0.5, -0.38), frameon=False)
fig.tight_layout()
plt.show()

## 5. Approach

### Methods and libraries
- **Data handling:** `datasets` (loading from Hugging Face), `pandas`.
- **Segmentation:** rule-based splitting of each solution into steps of ≥ 15 words (`re`, string methods).
- **Lexical analysis:** regular-expression counts of hand-picked exploration and exploitation markers, normalised per 1,000 words.
- **Text representation:** bag-of-words **TF-IDF** (`scikit-learn` `TfidfVectorizer`) per step.
- **Similarity:** **cosine similarity** between step vectors, turned into two trajectory measures (step distance, novelty).
- **Statistics:** paired **Wilcoxon signed-rank test** (`scipy`), and a shuffled-order baseline.
- **Visualisation:** `matplotlib`.
- **Planned next:** sentence embeddings (e.g. `sentence-transformers`) instead of TF-IDF, clustering steps into "strategies", and a hand-labelled validation sample.

### Assumptions and considerations
- **What is a "step"?** I assume a chunk of lines with ≥ 15 words is one reasoning step. This is a choice: other thresholds, sentences or paragraphs would give different numbers. Very short chunks (often just a formula) would make the distance measures noisy.
- **Representation.** TF-IDF only sees which words are shared, not meaning. Two steps can express the same idea in different words (high distance) or reuse the problem's vocabulary while doing different things (low distance). Embeddings would address this. I removed LaTeX commands and digits, and dropped steps with no remaining words, because formulas would otherwise dominate or create empty vectors.
- **Marker lists are crude.** Words like "try" or "suppose" are also used in ordinary proof writing ("suppose for contradiction"), and the lists are my own. They are transparent, but not validated.
- **Length is a confound.** Solutions differ a lot in length, which affects counts and the number of steps. I normalise per 1,000 words and average over steps, and report length in the comparison table so differences can be read in that light.
- **Pairing by problem.** Each problem has both human and model solutions, so I compare within problems rather than pooling, which controls for problem difficulty and topic. The four human solutions per problem are averaged, since they are not independent of each other.
- **Small sample.** 50 problems and only 50 model solutions: results are exploratory and p-values should be read with caution.
- **Interpretation limit.** The humans were asked to imitate a model's style, so the comparison is between *performed* human reasoning and model reasoning, not spontaneous human thinking.

## Preliminary findings
*(Fill in after running: e.g. "Human solutions had a higher/lower exploration share in X of 50 problems (p = …). Real step order gave lower distances than shuffled order, which suggests consecutive steps are related and the metric picks up the flow of the argument.")*